# dataloader-batching — ex2: drop_last semantics — count batches and verify size invariants

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `dataloader-batching`. Running the final beacon cell reports progress against the `PyTorch: DataLoader batching` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: DataLoader batching` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dataloader-batching`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dataloader-batching"
DD_SUBTOPIC = "PyTorch: DataLoader batching"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## DataLoader — `drop_last` semantics

If `len(dataset) % batch_size != 0`, the final batch is **smaller** than the others. The `drop_last` kwarg controls what to do with it:

- `drop_last=False` (default): keep the partial batch. `len(loader) = ceil(N / B)`, last batch has `N % B` items.
- `drop_last=True`: discard it. `len(loader) = N // B`, every batch is exactly `B` items.

When does `drop_last=True` matter? **BatchNorm with very small final batches** (variance estimate is unstable on tiny batches), and **DDP** (uneven batch sizes across ranks cause hang). Otherwise keep the partial batch — losing a few samples per epoch is wasteful.

The previous drill (ex1) wrapped a TensorDataset and iterated batches with the shuffle conventions. This drill targets **`drop_last`** specifically — counting batches in both modes and verifying batch-size invariants.

### Exercise 2 — drop_last semantics — count batches and verify size invariants

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `DataLoader(..., drop_last=True/False)` to a dataset whose size does not divide the batch size evenly, and count the resulting batches to confirm the partial-batch handling rule.
> Keywords: dataloader, drop-last, partial-batch, batch-count
> ```

**KCs targeted:** `dataloader-wraps-dataset`, `dataloader-drop-last-discards-partial`

Implement `ex2_drop_last_counts(N, batch_size)`. Build a `TensorDataset(t.arange(N).float())` and wrap it TWICE — once with `drop_last=True`, once with `drop_last=False`. Iterate each loader fully, collecting the batch sizes into a list. Return:

```python
{
  'with_drop':    [b0_size, b1_size, ...],  # drop_last=True
  'without_drop': [b0_size, b1_size, ...],  # drop_last=False
}
```

Set `shuffle=False` on both loaders so the iteration order is deterministic.

**The verification rules** (test will check these):
1. `len(with_drop) == N // batch_size`  (integer floor).
2. Every entry of `with_drop` equals `batch_size`.
3. `len(without_drop) == math.ceil(N / batch_size)`.
4. All entries of `without_drop` except possibly the LAST equal `batch_size`.
5. The last entry of `without_drop` equals `N % batch_size` if `N % batch_size != 0`, else `batch_size`.


In [ ]:
def ex2_drop_last_counts(N: int, batch_size: int) -> dict:
    """Return {with_drop: [...], without_drop: [...]} batch-size lists."""
    raise NotImplementedError()


def _test_ex2():
    import math
    from torch.utils.data import TensorDataset, DataLoader

    # Case 1 — non-dividing: 10 samples, batch_size 3.
    result = ex2_drop_last_counts(N=10, batch_size=3)
    assert set(result.keys()) == {'with_drop', 'without_drop'}, f'bad keys: {result.keys()}'
    wd, nd = result['with_drop'], result['without_drop']
    assert wd == [3, 3, 3], f'with_drop expected [3,3,3], got {wd}'
    assert nd == [3, 3, 3, 1], f'without_drop expected [3,3,3,1], got {nd}'

    # Case 2 — clean division: 8 samples, batch_size 4.
    r2 = ex2_drop_last_counts(N=8, batch_size=4)
    assert r2['with_drop'] == [4, 4], f'with_drop expected [4,4], got {r2[chr(39)+"with_drop"+chr(39)]}'
    assert r2['without_drop'] == [4, 4], f'without_drop expected [4,4], got {r2[chr(39)+"without_drop"+chr(39)]}'

    # Case 3 — single partial batch only: 3 samples, batch_size 5.
    r3 = ex2_drop_last_counts(N=3, batch_size=5)
    assert r3['with_drop'] == [], f'with_drop expected [], got {r3[chr(39)+"with_drop"+chr(39)]}'
    assert r3['without_drop'] == [3], f'without_drop expected [3], got {r3[chr(39)+"without_drop"+chr(39)]}'

    # Case 4 — confirm counts vs the closed-form rules on a bigger setting.
    N, B = 127, 16
    r4 = ex2_drop_last_counts(N=N, batch_size=B)
    assert len(r4['with_drop']) == N // B, f'with_drop count: expected {N//B}, got {len(r4[chr(39)+"with_drop"+chr(39)])}'
    assert len(r4['without_drop']) == math.ceil(N / B), f'without_drop count: expected {math.ceil(N/B)}, got {len(r4[chr(39)+"without_drop"+chr(39)])}'
    assert all(sz == B for sz in r4['with_drop']), 'with_drop entries must all equal batch_size'
    assert all(sz == B for sz in r4['without_drop'][:-1]), 'without_drop non-final entries must equal batch_size'
    tail = r4['without_drop'][-1]
    expected_tail = N % B if N % B != 0 else B
    assert tail == expected_tail, f'tail batch wrong: expected {expected_tail}, got {tail}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_drop_last_counts(N: int, batch_size: int) -> dict:
    from torch.utils.data import TensorDataset, DataLoader
    ds = TensorDataset(t.arange(N).float())
    with_drop = [b[0].shape[0] for b in DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=True)]
    without_drop = [b[0].shape[0] for b in DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False)]
    return {'with_drop': with_drop, 'without_drop': without_drop}
```

**When `drop_last=True` is mandatory.** Synchronous distributed training (DDP) hangs if different ranks see different batch counts. Setting `drop_last=True` aligns them. BatchNorm modules with `track_running_stats=True` also misbehave on tiny final batches because the running variance estimate goes haywire on N=1 or N=2.

**When it's wasteful.** Validation/test loops should set `drop_last=False` — you want every sample in your metrics.

**Difference from ex1.** ex1 set up train (shuffle=True) and test (shuffle=False) loaders with default `drop_last`. ex2 zeroes in on the `drop_last` knob and verifies the floor vs ceil batch-count formula.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()